# E5-B — Semantic vs. non-semantic preparation: controlled end-to-end audit
**TalTech / Energy Informatics — reproducible research notebook**

The notebook evaluates **two explicitly separated comparisons**:

- **Full-data analytical preparation**: conventional CSV vs. existing SPARQL-extracted semantic observations, each subjected to the **same independently executed** 15-minute grid, collision resolution and short-gap interpolation. Reference prepared data are compared where available.
- **Bounded end-to-end RDF workflow**: on the *same subset of raw observations*, execute CSV validation/preparation and **RDF construction → SHACL validation (if installed) → SPARQL extraction → preparation**, with identical rules. Measure each stage separately. It is a controlled subset diagnostic, **not** proof of full-scale runtime benefit.

**Interpretation rule:** An RDF/SPARQL dataset identical to a CSV-produced dataset does not establish a prediction-quality advantage. More validation constraints ≠ intrinsic semantic superiority. Do not call the exported-file reading time an end-to-end semantic runtime.

No original sources are changed. All outputs go to `E5_B_results/`.

In [ ]:
from pathlib import Path
import io, json, zipfile, hashlib, time, sys, warnings
from contextlib import contextmanager
import numpy as np
import pandas as pd

# Choose either the local `outputs` directory or the original archive.
OUTPUTS_SOURCE = Path('outputs')
if not OUTPUTS_SOURCE.exists():
    for candidate in [Path('outputs(1).zip')]:
        if candidate.exists(): OUTPUTS_SOURCE = candidate; break
RESULT_DIR = Path('E5_B_results')
RESULT_DIR.mkdir(exist_ok=True, parents=True)

# Full analytical reconciliation is enabled. If memory is constrained, set False.
RUN_FULL_PREPARATION = True
# Bounded end-to-end RDF construction + SHACL + SPARQL benchmark.
RUN_RDF_SUBSET = True
RDF_SUBSET_N = 1500   # for RDF/SHACL, increase cautiously after a successful run
TOLERANCE = 1e-5
REPEATS = 3

print('Source:', OUTPUTS_SOURCE.resolve())
print('Full analytical preparation:',RUN_FULL_PREPARATION, '| RDF subset:',RDF_SUBSET_N)


In [ ]:
class OutputStore:
    def __init__(self, path):
        self.path=Path(path)
        if self.path.is_file():
            self.zip=zipfile.ZipFile(self.path)
            self.names=set(self.zip.namelist())
        elif self.path.is_dir():
            self.zip=None
        else:
            raise FileNotFoundError(f'No outputs directory or ZIP found: {self.path}')
    @contextmanager
    def open(self, relative):
        rel=str(relative).replace('\\','/').lstrip('/')
        if self.zip:
            options=[rel, 'outputs/'+rel]
            match=next((p for p in options if p in self.names), None)
            if match is None: raise FileNotFoundError(f'{rel} not found in archive')
            with self.zip.open(match) as f: yield f
        else:
            file=self.path/rel
            if not file.exists(): raise FileNotFoundError(file)
            with file.open('rb') as f: yield f
    def csv(self, rel, **kwargs):
        with self.open(rel) as f: return pd.read_csv(f,**kwargs)
    def json(self,rel):
        with self.open(rel) as f: return json.load(f)

store=OutputStore(OUTPUTS_SOURCE)
RAW='01_extraction/02_selected_raw_observations_long.csv'
EXPORTED='04_sparql/03_ml_extraction_long_traceable.csv'
REF_WIDE='05_ml_dataset/05_synchronized_wide_all_slots.csv'
REF_COMPLETE='05_ml_dataset/06_ai_ready_complete_case.csv'
REF_PROTOCOL='05_ml_dataset/08_preparation_protocol_manifest.csv'
ref_manifest=store.csv(REF_PROTOCOL)
display(ref_manifest)
assert ref_manifest.loc[ref_manifest.parameter=='slot_projection','value'].iloc[0]=='nearest 15-minute slot'
assert ref_manifest.loc[ref_manifest.parameter=='collision_aggregation','value'].iloc[0]=='arithmetic mean'


## 1. Normalize sources and verify original-to-semantic-export equivalence
Matching keys: `(uuid, exact UTC timestamp, variable)`. Numeric equality is evaluated at a prespecified absolute tolerance of `1e-5` (as in E5-A). This is **artifact equivalence**, not a measurement of RDF construction effort.

In [ ]:
def normalize_source(source, semantic=False):
    if semantic:
        out=source[['uuid','timestamp','variable','value','unit']].copy()
    else:
        out=source[['device_id','timestamp','variable','value','unit']].rename(columns={'device_id':'uuid'}).copy()
    out['timestamp']=pd.to_datetime(out['timestamp'],utc=True,errors='raise')
    out['value']=pd.to_numeric(out['value'],errors='coerce')
    out['variable']=out['variable'].str.strip().str.lower()
    return out

start=time.perf_counter(); raw_original=store.csv(RAW); load_raw=time.perf_counter()-start
start=time.perf_counter(); sem_original=store.csv(EXPORTED); load_sem_export=time.perf_counter()-start
raw=normalize_source(raw_original); semantic=normalize_source(sem_original,True)
keys=['uuid','timestamp','variable']
assert not raw.duplicated(keys).any(), 'Duplicate raw exact keys; reconcile before merging'
assert not semantic.duplicated(keys).any(), 'Duplicate semantic exact keys; reconcile before merging'
merged=raw.merge(semantic,on=keys,how='outer',indicator=True,suffixes=('_raw','_rdf'),validate='one_to_one')
common=merged['_merge'].eq('both')
absdiff=(merged.loc[common,'value_raw']-merged.loc[common,'value_rdf']).abs()
recon={
 'raw_rows':int(len(raw)), 'semantic_export_rows':int(len(semantic)),
 'exact_key_matches':int(common.sum()),
 'raw_only':int(merged['_merge'].eq('left_only').sum()),
 'rdf_only':int(merged['_merge'].eq('right_only').sum()),
 'value_matches_at_tolerance':int((absdiff<=TOLERANCE).sum()),
 'max_absolute_value_difference':float(absdiff.max()),
 'unit_disagreements':int((merged.loc[common,'unit_raw'].to_numpy()!=merged.loc[common,'unit_rdf'].to_numpy()).sum()),
 'raw_read_s':load_raw, 'precomputed_semantic_export_read_s':load_sem_export,
}
(RESULT_DIR/'01_source_reconciliation.json').write_text(json.dumps(recon,indent=2))
display(pd.DataFrame([recon]).T.rename(columns={0:'value'}))


## 2. Independent, identical analytical preprocessing on both sources

The documented preparation uses **nearest** 15-minute grid projection, **arithmetic mean** collision resolution, and **time interpolation of internal missing runs only when the entire run is at most four 15-minute slots**. Processing is carried out separately on raw and SPARQL-extracted data. Transformations are identical and are not learned from test observations.

In [ ]:
def to_grid(data, start=None, end=None):
    d=data[['uuid','timestamp','variable','value']].copy()
    d['slot']=d['timestamp'].dt.round('15min')
    if start is not None: d=d[d.slot>=start]
    if end is not None: d=d[d.slot<=end]
    collisions=d.groupby(['uuid','variable','slot'],sort=True).agg(value=('value','mean'),raw_count=('value','size')).reset_index()
    return collisions

def prepare(data, start, end):
    grid=to_grid(data,start,end)
    uuids=sorted(data.uuid.unique())
    grid_index=pd.date_range(start,end,freq='15min',tz='UTC')
    variables=['temperature','relative_humidity','co2']
    pieces=[]
    n_interpolated=0
    for uuid in uuids:
        part=grid[grid.uuid==uuid].pivot(index='slot',columns='variable',values='value')
        part=part.reindex(index=grid_index,columns=variables)
        for var in variables:
            ser=part[var]; missing=ser.isna()
            # Run ID for maximal contiguous missing sequences.
            group_id=missing.ne(missing.shift(fill_value=False)).cumsum()
            lengths=missing.groupby(group_id).transform('sum')
            small=missing & lengths.le(4)
            candidate=ser.interpolate(method='time',limit_area='inside')
            filled=ser.where(~small,candidate)
            n_interpolated+=int((missing & filled.notna()).sum())
            part[var]=filled
        part.index.name='timestamp'
        part=part.reset_index()
        part.insert(0,'uuid',uuid)
        part['complete_trhco2']=part[variables].notna().all(axis=1)
        pieces.append(part)
    wide=pd.concat(pieces,ignore_index=True)
    complete=wide.loc[wide.complete_trhco2,['uuid','timestamp']+variables].copy()
    return wide,complete,{'grid_buckets':len(grid),'slots_with_collisions':int((grid.raw_count>1).sum()),'interpolated_cells':n_interpolated,'complete_rows':len(complete)}

interval_start=pd.Timestamp('2025-02-07 12:15:00+00:00')
interval_end=pd.Timestamp('2025-06-27 17:45:00+00:00')

if RUN_FULL_PREPARATION:
    a=time.perf_counter(); raw_wide,raw_complete,raw_stats=prepare(raw,interval_start,interval_end)
    raw_prepare_s=time.perf_counter()-a
    a=time.perf_counter(); rdf_wide,rdf_complete,rdf_stats=prepare(semantic,interval_start,interval_end)
    rdf_prepare_s=time.perf_counter()-a
    prep_summary=pd.DataFrame([{'branch':'Direct CSV','read_s':load_raw,'prepare_s':raw_prepare_s,**raw_stats},
                               {'branch':'Existing RDF/SPARQL export','read_s':load_sem_export,'prepare_s':rdf_prepare_s,**rdf_stats}])
    display(prep_summary)
    prep_summary.to_csv(RESULT_DIR/'02_analytic_preparation_stats.csv',index=False)
else:
    print('Full preparation disabled; full-data comparisons will not be claimed.')


## 3. Compare both outputs with one another and with the archived wide/complete datasets

The archive is a **reference**, not the source of reconstructed values. Small floating-point differences are tested explicitly; any material disagreement is retained in an export rather than silently corrected.

In [ ]:
def compare_prepared(left,right,label):
    vals=['temperature','relative_humidity','co2']
    key=['uuid','timestamp']
    assert not left.duplicated(key).any() and not right.duplicated(key).any()
    joined=left[key+vals].merge(right[key+vals],on=key,how='outer',indicator=True,validate='one_to_one',suffixes=('_a','_b'))
    both=joined._merge.eq('both'); mismatches={}
    for v in vals:
        a=joined.loc[both,v+'_a'];b=joined.loc[both,v+'_b'];
        diff=(a-b).abs();both_na=a.isna()&b.isna()
        mismatches[v]=int((~both_na & ~(diff<=TOLERANCE)).sum())
    result={'comparison':label,'left_rows':len(left),'right_rows':len(right),
            'matched_keys':int(both.sum()),'left_only':int(joined._merge.eq('left_only').sum()),
            'right_only':int(joined._merge.eq('right_only').sum()),
            **{f'{v}_mismatch':c for v,c in mismatches.items()}}
    return result

if RUN_FULL_PREPARATION:
    reference_wide=store.csv(REF_WIDE)
    reference_complete=store.csv(REF_COMPLETE)
    for frame in [reference_wide,reference_complete]: frame['timestamp']=pd.to_datetime(frame['timestamp'],utc=True)
    comparisons=[compare_prepared(raw_wide,rdf_wide,'CSV vs RDF export — complete time grid'),
                 compare_prepared(raw_complete,rdf_complete,'CSV vs RDF export — complete cases'),
                 compare_prepared(rdf_wide,reference_wide,'Recomputed RDF vs archived time grid'),
                 compare_prepared(rdf_complete,reference_complete,'Recomputed RDF vs archived complete cases')]
    comparison_table=pd.DataFrame(comparisons)
    display(comparison_table)
    comparison_table.to_csv(RESULT_DIR/'03_analytic_equivalence_checks.csv',index=False)
    print('Important: inspect discrepancies before making manuscript claims.')


## 4. Explicit common-rule validation on identical data copies

The same rules for required UUID/time/variable/value/unit and valid variable–unit pairs are evaluated in each branch. URI checks are **representation-specific**, and are not scored as a win against CSV (which has no URI columns).

In [ ]:
UNIT_MAP={'temperature':'C','relative_humidity':'%','co2':'ppm'}
def validate_common(df):
    d=df.copy(); expected=d['variable'].map(UNIT_MAP)
    return pd.DataFrame({'missing_uuid':d['uuid'].isna()|d['uuid'].astype(str).str.strip().eq(''),
                         'missing_timestamp':d['timestamp'].isna(),
                         'unknown_variable':expected.isna(),
                         'invalid_unit':d['unit'].ne(expected) & expected.notna(),
                         'missing_value':d['value'].isna()})

def inject_faults(df):
    d=df.head(30).copy().reset_index(drop=True)
    d.loc[0,'uuid']=None;d.loc[1,'timestamp']=pd.NaT;d.loc[2,'variable']='unknown_variable'
    d.loc[3,'unit']='invalid';d.loc[4,'value']=np.nan
    return d

raw_fault=inject_faults(raw)
rdf_fault=inject_faults(semantic)
common_faults=pd.DataFrame([{'branch':'CSV','type':k,'flagged':int(v)} for k,v in validate_common(raw_fault).sum().items()]+[
                           {'branch':'RDF-export','type':k,'flagged':int(v)} for k,v in validate_common(rdf_fault).sum().items()])
display(common_faults)
common_faults.to_csv(RESULT_DIR/'04_common_fault_checks.csv',index=False)


## 5. Bounded **actual** RDF → validation → SPARQL → preparation path

Create an RDF graph from raw records, validate common fields, optionally execute SHACL, query the graph, then compare the recovered values and prepared snapshots against CSV for the **same N records**. Graph construction and SPARQL are timed separately. SHACL is not reported as executed unless `pyshacl` is installed and completes. This synthetic graph is a *controlled minimal semantic model*, not a reconstruction of the 2.9-million-triple production ontology.

In [ ]:
import rdflib
from rdflib import Graph,URIRef,Literal,Namespace
from rdflib.namespace import RDF,XSD
try:
    from pyshacl import validate as shacl_validate
    SHACL_AVAILABLE=True
except ImportError:
    SHACL_AVAILABLE=False
EX=Namespace('https://example.org/e5b/')
SOSA=Namespace('http://www.w3.org/ns/sosa/')

def construct_graph(frame):
    graph=Graph();graph.bind('ex',EX);graph.bind('sosa',SOSA)
    for i,row in frame.reset_index(drop=True).iterrows():
        obs=EX[f'observation/{i}']
        point=EX[f'point/{row.uuid}/{row.variable}']
        graph.add((obs,RDF.type,SOSA.Observation))
        graph.add((obs,EX.operationalUUID,Literal(str(row.uuid))))
        graph.add((obs,EX.variable,Literal(str(row.variable))))
        graph.add((obs,EX.unit,Literal(str(row.unit))))
        graph.add((obs,EX.point,point))
        graph.add((obs,SOSA.resultTime,Literal(row.timestamp.isoformat(),datatype=XSD.dateTime)))
        graph.add((obs,SOSA.hasSimpleResult,Literal(float(row.value),datatype=XSD.double)))
    return graph

SHAPES='''@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix sosa: <http://www.w3.org/ns/sosa/> .
@prefix ex: <https://example.org/e5b/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
ex:ObservationShape a sh:NodeShape ; sh:targetClass sosa:Observation ;
    sh:property [sh:path ex:operationalUUID; sh:minCount 1; sh:maxCount 1; sh:datatype xsd:string] ;
    sh:property [sh:path ex:variable; sh:minCount 1; sh:maxCount 1; sh:in ("temperature" "relative_humidity" "co2")] ;
    sh:property [sh:path ex:unit; sh:minCount 1; sh:maxCount 1] ;
    sh:property [sh:path ex:point; sh:minCount 1; sh:maxCount 1; sh:nodeKind sh:IRI] ;
    sh:property [sh:path sosa:resultTime; sh:minCount 1; sh:maxCount 1; sh:datatype xsd:dateTime] ;
    sh:property [sh:path sosa:hasSimpleResult; sh:minCount 1; sh:maxCount 1; sh:datatype xsd:double] .
'''

query='''PREFIX ex: <https://example.org/e5b/>
PREFIX sosa: <http://www.w3.org/ns/sosa/>
SELECT ?uuid ?timestamp ?variable ?value ?unit ?point WHERE {
  ?obs a sosa:Observation ; ex:operationalUUID ?uuid ;
    ex:variable ?variable ; ex:unit ?unit ;
    ex:point ?point ; sosa:resultTime ?timestamp ; sosa:hasSimpleResult ?value .
}'''

subset_raw=raw.sort_values(['uuid','timestamp','variable'],kind='mergesort').head(RDF_SUBSET_N).copy()
subset_times=[]
if RUN_RDF_SUBSET:
    for rep in range(REPEATS):
        t=time.perf_counter(); graph=construct_graph(subset_raw); build_s=time.perf_counter()-t
        t=time.perf_counter(); common_violations=int(validate_common(subset_raw).any(axis=1).sum()); conventional_validate_s=time.perf_counter()-t
        if SHACL_AVAILABLE:
            shapes=Graph().parse(data=SHAPES,format='turtle')
            t=time.perf_counter(); conforms,report,_=shacl_validate(graph,shacl_graph=shapes,inference='none',abort_on_first=False)
            shacl_s=time.perf_counter()-t
            if not conforms: print(str(report)[:1000])
        else:
            conforms=None;shacl_s=None
        t=time.perf_counter();rows=list(graph.query(query));sparql_s=time.perf_counter()-t
        subset_times.append({'repeat':rep,'rows':len(subset_raw),'triples':len(graph),
                             'csv_validation_s':conventional_validate_s,'rdf_construction_s':build_s,
                             'shacl_s':shacl_s,'sparql_s':sparql_s,
                             'common_csv_violations':common_violations,'shacl_conforms':conforms,
                             'shacl_executed':SHACL_AVAILABLE})
    bench=pd.DataFrame(subset_times)
    display(bench)
    bench.to_csv(RESULT_DIR/'05_subset_end_to_end_stage_times.csv',index=False)
    from_rdf=pd.DataFrame([{'uuid':str(r.uuid),'timestamp':str(r.timestamp),'variable':str(r.variable),
                            'value':float(r.value),'unit':str(r.unit)} for r in rows])
    from_rdf['timestamp']=pd.to_datetime(from_rdf['timestamp'],utc=True)
    # IDs and numerical matching on the original observations.
    left=subset_raw.sort_values(['uuid','timestamp','variable']).reset_index(drop=True)
    right=from_rdf.sort_values(['uuid','timestamp','variable']).reset_index(drop=True)
    assert left[['uuid','timestamp','variable']].equals(right[['uuid','timestamp','variable']])
    subset_rec={'raw_rows':len(left),'sparql_rows':len(right),
                'maximum_absolute_value_difference':float((left.value-right.value).abs().max()),
                'unit_mismatches':int((left.unit!=right.unit).sum()),
                'shacl_executed':SHACL_AVAILABLE}
    (RESULT_DIR/'06_subset_query_roundtrip.json').write_text(json.dumps(subset_rec,indent=2))
    print('Subset roundtrip:',subset_rec)
else: print('RDF subset experiment disabled.')


## 6. Interpret the evidence without unsupported causal claims

A proper **full production** time comparison would require replayable construction code and equivalent validation/extraction functionality on the original raw data, with separate, repeated timings. A minimal synthetic RDF graph is intentionally *not* substituted for the production Brick/SOSA graph. The dataset preparation comparison is fully executed on both existing raw and semantic-exported files, whereas construction/SHACL/SPARQL are measured only on the bounded synthetic graph.

⚠️ The synthetic URI format is controlled for this notebook; it does not verify IFC links, spatial relations, production ontology shape completeness, or manual effort.

In [ ]:
summary={
 'experiment':'E5-B controlled CSV vs RDF end-to-end audit',
 'raw_source':RAW,'semantic_export':EXPORTED,
 'full_preparation_executed':bool(RUN_FULL_PREPARATION),
 'bounded_rdf_workflow_executed':bool(RUN_RDF_SUBSET),
 'bounded_rdf_observations':int(RDF_SUBSET_N) if RUN_RDF_SUBSET else 0,
 'shacl_executed':bool(SHACL_AVAILABLE and RUN_RDF_SUBSET),
 'common_preparation_rules':'nearest 15-min bin; mean collision; internal missing runs <=4 grid slots time-interpolated',
 'limitation_1':'Full raw→production Brick graph construction is not replayed in this notebook.',
 'limitation_2':'Full production SHACL/SPARQL runtime is not compared with CSV end-to-end.',
 'limitation_3':'No causal RDF advantage for prediction, manual effort or interoperability is established.',
 'limitation_4':'Synthetic bounded graph uses SOSA + custom terms; it is not the full production ontology.'}
if RUN_FULL_PREPARATION:
    summary['full_reconciliation_results']=comparisons
(RESULT_DIR/'07_scope_limitations_manifest.json').write_text(json.dumps(summary,indent=2,default=str))
print('Files exported to:',RESULT_DIR.resolve())
print('Report only results actually executed; do not add superiority claims without stronger controls.')
